In [156]:
import numpy as np 
import pandas as pd 

In [157]:
orders = pd.read_csv('/kaggle/input/datasets/pratik150507/raawwww/olist_orders_dataset.csv')
items = pd.read_csv('/kaggle/input/datasets/pratik150507/raawwww/olist_order_items_dataset.csv')
products = pd.read_csv('/kaggle/input/datasets/pratik150507/raawwww/olist_products_dataset.csv')
customers = pd.read_csv('/kaggle/input/datasets/pratik150507/raawwww/olist_customers_dataset.csv')
payments = pd.read_csv('/kaggle/input/datasets/pratik150507/raawwww/olist_order_payments_dataset.csv')
reviews = pd.read_csv('/kaggle/input/datasets/pratik150507/raawwww/olist_order_reviews_dataset.csv')
translation = pd.read_csv('/kaggle/input/datasets/pratik150507/raawwww/product_category_name_translation.csv')

In [158]:
datasets = {
    "orders": orders,
    "items": items,
    "products": products,
    "customers": customers,
    "payments": payments,
    "reviews": reviews,
    "translation": translation
}

In [159]:
for name, df in datasets.items():
    print(f"\n{name.upper()} INFO:")
    print(df.info())


ORDERS INFO:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB
None

ITEMS INFO:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id     

In [160]:
for name, df in datasets.items():
    print(f"\n{name.upper()} Null Count:")
    print(df.isnull().sum())



ORDERS Null Count:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

ITEMS Null Count:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

PRODUCTS Null Count:
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

CUSTOMERS Null Count:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
cu

# ORDERS CLEANING


In [161]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")


In [162]:
orders["delivery_time_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

orders.loc[orders["delivery_time_days"] < 0, "delivery_time_days"] = np.nan

In [163]:
orders["is_delivered_clean"] = orders["order_delivered_customer_date"].notna().astype(int)

In [164]:
orders["delivery_time_days"] = np.where(
    orders["is_delivered_clean"] == 1,
    (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days,
    np.nan
)

In [165]:
orders["order_stage"] = np.select(
    [
        orders["order_delivered_customer_date"].notna(),
        orders["order_delivered_carrier_date"].notna(),
        orders["order_approved_at"].notna()
    ],
    [
        "delivered",
        "shipped_not_delivered",
        "approved_not_shipped"
    ],
    default="created_only"
)


In [166]:
orders["is_completed_order"] = (
    orders["order_stage"] == "delivered"
).astype(int)

In [167]:
orders["order_stage"].value_counts()

order_stage
delivered                96476
approved_not_shipped      1636
shipped_not_delivered     1183
created_only               146
Name: count, dtype: int64

In [168]:
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
delivery_time_days               2965
is_delivered_clean                  0
order_stage                         0
is_completed_order                  0
dtype: int64

# ITEMS CLEANING

In [169]:
items = items.copy()

items["shipping_limit_date"] = pd.to_datetime(
    items["shipping_limit_date"],
    errors="coerce"
)

items["item_total_value"] = items["price"] + items["freight_value"]
items["is_expensive_item"] = (items["price"] > items["price"].quantile(0.95)).astype(int)

items.isna().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
item_total_value       0
is_expensive_item      0
dtype: int64

# PRODUCTS CLEANING

In [170]:
products = products.copy()

products["product_category_name"] = products["product_category_name"].fillna("unknown")

num_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for col in num_cols:
    products[col] = products[col].fillna(products[col].median())

products["product_volume_cm3"] = (
    products["product_length_cm"] *
    products["product_height_cm"] *
    products["product_width_cm"]
)


products.isna().sum()


product_id                    0
product_category_name         0
product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
product_weight_g              0
product_length_cm             0
product_height_cm             0
product_width_cm              0
product_volume_cm3            0
dtype: int64

# CUSTOMERS CLEANING

In [171]:
customers = customers.copy()

customers = customers.drop_duplicates()

customers["state_city"] = customers["customer_state"] + "_" + customers["customer_city"]

customers.isna().sum()


customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
state_city                  0
dtype: int64

# PAYMENTS CLEANING

In [172]:
payments = payments.copy()

payments["payment_value_per_installment"] = (
    payments["payment_value"] /
    payments["payment_installments"].replace(0, np.nan)
)

payments["payment_value_per_installment"] = payments[
    "payment_value_per_installment"
].fillna(payments["payment_value"])

payments["is_credit_card"] = (payments["payment_type"] == "credit_card").astype(int)

payments["is_full_payment"] = (payments["payment_installments"] == 1).astype(int)

payments.isna().sum()


order_id                         0
payment_sequential               0
payment_type                     0
payment_installments             0
payment_value                    0
payment_value_per_installment    0
is_credit_card                   0
is_full_payment                  0
dtype: int64

# REVIEWS CLEANING

In [173]:
reviews = reviews.copy()

reviews["review_creation_date"] = pd.to_datetime(
    reviews["review_creation_date"],
    errors="coerce"
)

reviews["review_answer_timestamp"] = pd.to_datetime(
    reviews["review_answer_timestamp"],
    errors="coerce"
)

reviews["review_comment_title"] = reviews["review_comment_title"].fillna("no_title")
reviews["review_comment_message"] = reviews["review_comment_message"].fillna("no_comment")

def sentiment(score):
    if score >= 4:
        return "positive"
    elif score == 3:
        return "neutral"
    else:
        return "negative"

reviews["sentiment"] = reviews["review_score"].apply(sentiment)

reviews["is_negative_review"] = (reviews["review_score"] <= 2).astype(int)

reviews.isna().sum()


review_id                  0
order_id                   0
review_score               0
review_comment_title       0
review_comment_message     0
review_creation_date       0
review_answer_timestamp    0
sentiment                  0
is_negative_review         0
dtype: int64

# TRANSLATION CLEANING

In [174]:
translation = translation.copy()

translation = translation.drop_duplicates()

translation.columns = [
    "product_category_name",
    "product_category_name_english"
]

translation.isna().sum()


product_category_name            0
product_category_name_english    0
dtype: int64

In [175]:
datasets = {
    "orders": orders,
    "items": items,
    "products": products,
    "customers": customers,
    "payments": payments,
    "reviews": reviews,
    "translation": translation
}

for name, df in datasets.items():
    print(f"\n{name.upper()} SHAPE:", df.shape)
    print(df.isna().sum())



ORDERS SHAPE: (99441, 12)
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
delivery_time_days               2965
is_delivered_clean                  0
order_stage                         0
is_completed_order                  0
dtype: int64

ITEMS SHAPE: (112650, 9)
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
item_total_value       0
is_expensive_item      0
dtype: int64

PRODUCTS SHAPE: (32951, 10)
product_id                    0
product_category_name         0
product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
product_weight_g              0
product_length_cm  

In [176]:
OUTPUT_PATH = "/kaggle/working/"

In [177]:
orders.to_csv(f"{OUTPUT_PATH}/orders_clean.csv", index=False)
items.to_csv(f"{OUTPUT_PATH}/items_clean.csv", index=False)
products.to_csv(f"{OUTPUT_PATH}/products_clean.csv", index=False)
customers.to_csv(f"{OUTPUT_PATH}/customers_clean.csv", index=False)
payments.to_csv(f"{OUTPUT_PATH}/payments_clean.csv", index=False)
reviews.to_csv(f"{OUTPUT_PATH}/reviews_clean.csv", index=False)
translation.to_csv(f"{OUTPUT_PATH}/translation_clean.csv", index=False)

In [178]:
df_master = orders.copy()

df_master = df_master.merge(customers, on="customer_id", how="left")
df_master = df_master.merge(items, on="order_id", how="left")
df_master = df_master.merge(products, on="product_id", how="left")
df_master = df_master.merge(payments, on="order_id", how="left")
df_master = df_master.merge(reviews, on="order_id", how="left")
df_master = df_master.merge(translation, on="product_category_name", how="left")


In [179]:
df_master.columns = df_master.columns.str.lower()
df_master = df_master.drop_duplicates()


In [180]:
df_master.columns = df_master.columns.str.lower()
df_master = df_master.drop_duplicates()

In [181]:
df_master.to_csv(f"{OUTPUT_PATH}/master_dataset.csv", index=False)